## Système de recommandation basé sur les profils utilisateurs

Ce notebook propose un système de recommandation d’images personnalisé pour chaque utilisateur, en se basant sur leurs préférences déclarées.

### Méthode utilisée : filtrage basé sur le contenu
Chaque image du dataset est comparée au profil utilisateur selon :
- Les couleurs dominantes de l’image
- Le format de l’image
- L’orientation de l’image
- Un bonus si l’image a déjà été aimée par l’utilisateur

Un **score de similarité** est calculé pour chaque image. Les 5 images ayant le score le plus élevé sont proposées à l’utilisateur.

Ce système constitue une première version d’un moteur de recommandation personnalisé basé sur des attributs visuels simples.

In [ ]:
import json
import pandas as pd
from collections import Counter

ANNOTATION_FILE = "../annotation/JSON/annotations.json"
USER_PROFILES_FILE = "../annotation/JSON/user_profiles.json"

# Charger les annotations et les profils utilisateurs
def load_json(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return json.load(f)

annotations = load_json(ANNOTATION_FILE)
user_profiles = load_json(USER_PROFILES_FILE)

# Convertir les annotations en DataFrame exploitable
data = []
for filename, info in annotations.items():
    meta = info["metadata"]
    data.append({
        "filename": filename,
        "format": meta.get("format"),
        "orientation": (
            "portrait" if meta.get("size", [0, 0])[1] > meta.get("size", [0, 0])[0]
            else "landscape" if meta.get("size", [0, 0])[0] > meta.get("size", [0, 0])[1]
            else "square"
        ),
        "colors": info.get("dominant_colors", [])
    })

df_images = pd.DataFrame(data)

# Fonction de similarité entre un utilisateur et une image
def compute_similarity(user, image):
    score = 0

    # Couleurs : intersection entre préférences et couleurs dominantes
    user_colors = set(user.get("preferred_colors", []))
    image_colors = set(image["colors"])
    color_overlap = len(user_colors.intersection(image_colors))
    score += color_overlap * 3  # poids fort pour les couleurs

    # Format
    if image["format"] == user.get("preferred_format"):
        score += 1

    # Orientation
    if image["orientation"] == user.get("preferred_orientation"):
        score += 2

    # Bonus si déjà likée
    if image["filename"] in user.get("liked_images", []):
        score += 5

    return score

# Géner des recommandations pour chaque utilisateur
def generate_recommendations(user_profiles, df_images, top_n=5):
    all_recommendations = {}

    for user in user_profiles:
        scored_images = []
        for _, row in df_images.iterrows():
            score = compute_similarity(user, row)
            if score > 0:
                scored_images.append((row["filename"], score))

        # Trier les images par score décroissant
        scored_images.sort(key=lambda x: x[1], reverse=True)
        top_images = [img for img, _ in scored_images[:top_n]]
        all_recommendations[user["id"]] = top_images

    return all_recommendations

# Génération effective
recommendations = generate_recommendations(user_profiles, df_images, top_n=5)

# Affichage des recommandations
for user_id, images in recommendations.items():
    print(f"\nRecommandations pour {user_id} :")
    for img in images:
        print(f"  - {img}")


Recommandations pour user_1 :
  - Cityscape%20of%20San%20Jos%C3%A9%2C%20Costa%20Rica%20%28253552473%29.jpg
  - Bangui%20Shopping%20District.jpg
  - Heidelberg%20corr.jpg
  - Montaje%20de%20Antofagasta.jpg
  - Residenzschloss%20Darmstadt%20539-Gdh.JPG

Recommandations pour user_2 :
  - Damascus%2C%20Syria%2C%20Panorama%20at%20sunset.jpg
  - Beograd%20collage.jpg
  - San%20Salvador%20Montage.jpg
  - Asmara2.jpg
  - Avenida%20Bolivar%20A%20Chavez.jpg

Recommandations pour user_3 :
  - Berliner%20Platz%20Guetersloh%20Proedous.jpg
  - AnkonPanorama.jpg
  - Dom%20R1.jpg
  - Luanda%20feb09%20ost01.jpg
  - Nairobi%2C%20view%20from%20KICC.JPG

Recommandations pour user_4 :
  - Beirutcity.jpg
  - Bangui%20City%20Centre.jpg
  - Downtown%20Lusaka.JPG
  - Kigali2018Cropped.jpg
  - Remscheid%20rathaus.jpg

Recommandations pour user_5 :
  - Damascus%2C%20Syria%2C%20Panorama%20at%20sunset.jpg
  - Bahrain%20World%20trade%20Center%20.jpg
  - Maputo%20seen%20from%20southeast%20-%20October%202006.jpg
  -